# Sparse Walker v1.1 — recurrence correctness control

This isolates three fixes only: **one fresh injection per event**, **duplicate concept coalescing before TopK**, and **no pursuit rewiring**. Everything else stays on the canonical ML-1M FullCE setup.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, subprocess, shutil, runpy, torch
REPO='/content/Sparsewalker'
BRANCH='agent/walker-v11-recurrence-fixes'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
SRC=f'{REPO}/src'; sys.path.insert(0,SRC)
for name in list(sys.modules):
    if name=='sparsewalker' or name.startswith('sparsewalker.'): del sys.modules[name]

from sparsewalker.models.sparsewalker import SparseWalker, _coalesced_topk

# 1) duplicate concepts must have their mass summed before pruning
ids=torch.tensor([[5,5,9,11]])
mass=torch.tensor([[.2,.3,.4,.1]])
oi,om=_coalesced_topk(ids,mass,3)
got={int(i):float(m) for i,m in zip(oi[0],om[0])}
assert len(set(oi[0].tolist()))==3, (oi,om)
assert abs(got[5]-.5)<1e-6 and abs(got[9]-.4)<1e-6 and abs(got[11]-.1)<1e-6, got
print('COALESCE_TEST OK',got,flush=True)

# 2) layers=2 means two graph hops, but only ONE fresh merge per timestep
class CountWalker(SparseWalker):
    def __init__(self,*a,**kw):
        super().__init__(*a,**kw); self.merge_calls=0
    def _merge(self,*a,**kw):
        self.merge_calls+=1
        return super()._merge(*a,**kw)
m=CountWalker(50,8,d=8,layers=2,side=8,h=4,active=4,top_side=2,degree=2)
x=torch.randint(1,51,(2,7))
_ = m.encode(x)
assert m.merge_calls==7, m.merge_calls
print('SINGLE_FRESH_INJECTION_TEST OK merge_calls=',m.merge_calls,'for 7 timesteps / 2 graph hops',flush=True)
del m,x

assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'bf16',torch.cuda.is_bf16_supported(),flush=True)
print('BRANCH',BRANCH,flush=True)

SCRIPT=f'{REPO}/experiments/run_ml1m_walker_v11.py'
sys.argv=[SCRIPT,'--seed','42','--max-epochs','50','--eval-every','5','--patience','20','--batch-size','128','--eval-batch-size','1024']
print('INPROCESS V1.1 RUN START',flush=True)
runpy.run_path(SCRIPT,run_name='__main__')
print('INPROCESS V1.1 RUN END',flush=True)


In [ ]:
import pandas as pd, json
from pathlib import Path
root=Path('/content/drive/MyDrive/sparsewalker_v11/ml1m/seed42')
if (root/'history.csv').exists():
    df=pd.read_csv(root/'history.csv')
    display(df[['epoch','loss','NDCG@10','HR@10','MRR@10','seconds','positions_per_s']])
    print('BEST VAL',df.loc[df['NDCG@10'].idxmax(),['epoch','NDCG@10']].to_dict())
if (root/'result.json').exists():
    print(json.dumps(json.loads((root/'result.json').read_text()),indent=2))
